In [2]:
import os
os.chdir(os.path.dirname(os.getcwd()))

In [3]:
import os
import sys
from argparse import ArgumentParser
import numpy as np
import pandas as pd
import time
import json
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold, train_test_split
from scipy.io import arff
from lightgbmlss.model import *
from lightgbmlss.distributions.Gaussian import *
from scipy.stats import norm
from utils.metrics import crps
import matplotlib.pyplot as plt
import uncertainty_toolbox as uct

In [9]:
np.random.seed(1)
mode = 'exp'

dataset_name_to_loader = {
    "Boston Housing": lambda: pd.read_csv(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/housing/housing.data",
        header=None,
        delim_whitespace=True,
    ),
    "Concrete Compression Strength": lambda: pd.read_excel(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls"
    ),
    "Energy Efficiency": lambda: pd.read_excel(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/00242/ENB2012_data.xlsx"
    ).iloc[:, :-1],
    "Kin8nm": lambda: pd.read_csv("ngboost/data/uci/kin8nm.csv"),
    "Naval Propulsion": lambda: pd.read_csv(
        "ngboost/data/uci/naval-propulsion.txt", delim_whitespace=True, header=None
    ).iloc[:, :-1],
    "Combined Cycle Power Plant": lambda: pd.read_excel("ngboost/data/uci/power-plant.xlsx"),
    "Protein Structure": lambda: pd.read_csv("ngboost/data/uci/protein.csv")[
        ["F1", "F2", "F3", "F4", "F5", "F6", "F7", "F8", "F9", "RMSD"]
    ],
    "Wine Quality Red": lambda: pd.read_csv(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv",
        delimiter=";",
    ),
    "Yacht Hydrodynamics": lambda: pd.read_csv(
        "http://archive.ics.uci.edu/ml/machine-learning-databases/00243/yacht_hydrodynamics.data",
        header=None,
        delim_whitespace=True,
    ),
    "Year Prediciton MSD": lambda: pd.read_csv("ngboost/data/uci/YearPredictionMSD.txt").iloc[:, ::-1],
}

dataset_list = ["Boston Housing", "Concrete Compression Strength", "Energy Efficiency", "Kin8nm", "Naval Propulsion", "Combined Cycle Power Plant", "Protein Structure", "Wine Quality Red", "Yacht Hydrodynamics", "Year Prediciton MSD"]

def run_single_arguement(run_seed):
    dset = dataset_list[int(run_seed)]
    args["dataset"] = dset
    y_true, lss_rmse, lss_nll, times = [], [], [], []
    lss_crps, lss_crps_cal, lss_crps_sha = [], [], []

    # Load dataset -- use last column as labela
    data = dataset_name_to_loader[args['dataset']]()
    X, y = data.iloc[:, :-1].values, data.iloc[:, -1].values

    print(f"== Dataset={args['dataset']} X.shape={str(X.shape)} {args['score']}/{args['distn']}")
    lgbm_rmse = []
    with open(f'logs/{mode}/{args["dataset"]}_opt_params.json') as pset:
        default_params = json.load(pset)
        
    X_trainall, X_test, y_trainall, y_test = train_test_split(
        X, y, test_size=0.2
    )        
        
    X_train, X_val, y_train, y_val = train_test_split(
        X_trainall, y_trainall, test_size=0.2
    )

    y_true += list(y_test.flatten())

    lgblss = LightGBMLSS(
        Gaussian(stabilization="None",
                response_fn = mode,
                loss_fn = "nll")
    )
    # Modify start values     
    lgblss.start_values = np.array([np.array(0.5) for _ in range(lgblss.dist.n_dist_param)])

    dtrain = lgb.Dataset(X_train, y_train)
    deval = lgb.Dataset(X_val, y_val)
    dtest = lgb.Dataset(X_test, y_test)
    # Training with early stopping
    evals_result = {}
    default_params['early_stopping'] = 20
    # Train Model with optimized hyperparameters
    gbm = lgblss.train(default_params, dtrain, 
                        num_boost_round = args["n_est"],
                        valid_sets = [dtrain, deval]
                        )

    # Best iteration
    print(f"Best iteration: {lgblss.booster.best_iteration}")

    full_train_data = lgb.Dataset(X_trainall, y_trainall)
    default_params['early_stopping'] = None

    final_gbm = lgblss.train(default_params, full_train_data, 
                        num_boost_round = lgblss.booster.best_iteration,
                    )
    # the final prediction for this fold
    forecast = lgblss.predict(X_test)
    return X_test, y_test, forecast['loc'], forecast['scale']

In [28]:
vsc_data = os.environ['VSC_DATA']
results = run_single_arguement(3)

== Dataset=Kin8nm X.shape=(8192, 8) MLE/Normal
[LightGBM] [Warning] Unknown parameter: opt_rounds
[1]	training's nll: 6700.38	valid_1's nll: 1672.67
Training until validation scores don't improve for 20 rounds
[2]	training's nll: 6209.71	valid_1's nll: 1550.08
[3]	training's nll: 5822.04	valid_1's nll: 1450.45
[4]	training's nll: 5503.07	valid_1's nll: 1368.7
[5]	training's nll: 5227.55	valid_1's nll: 1299.24
[6]	training's nll: 4980.19	valid_1's nll: 1237.65
[7]	training's nll: 4761.37	valid_1's nll: 1181.52
[8]	training's nll: 4566.2	valid_1's nll: 1132.48
[9]	training's nll: 4386.57	valid_1's nll: 1087.64
[10]	training's nll: 4220.94	valid_1's nll: 1045.95
[11]	training's nll: 4066.4	valid_1's nll: 1007.39
[12]	training's nll: 3919.73	valid_1's nll: 971.36
[13]	training's nll: 3786.83	valid_1's nll: 937.979
[14]	training's nll: 3659.46	valid_1's nll: 906.303
[15]	training's nll: 3538.29	valid_1's nll: 876.521
[16]	training's nll: 3423.08	valid_1's nll: 848.341
[17]	training's nll: 3

In [34]:
results[0][:,0].shape

(1639,)

In [30]:
f.shape

(206,)

In [31]:
std.shape

(206,)

In [32]:
y.shape

(206,)

In [33]:
x.shape

(206,)

In [36]:
# Set plot style
uct.viz.set_style()
uct.viz.update_rc("text.usetex", False)  # Set to True for system latex
uct.viz.update_rc("font.size", 14)  # Set font size
uct.viz.update_rc("xtick.labelsize", 14)  # Set font size for xaxis tick labels
uct.viz.update_rc("ytick.labelsize", 14)  # Set font size for yaxis tick labels

# Set random seed
np.random.seed(11)

# Generate synthetic predictive uncertainty results
n_obs = 650
f, std, y, x = results[2], results[3], results[1], results[0][:,0]

f, std, y, x =  np.array(f), np.array(std), np.array(y), np.array(x)

# Save figure (set to True to save)
savefig = True


def make_plots(pred_mean, pred_std, plot_save_str="row"):
    """Make set of plots."""

    ylims = [-3, 3]
    n_subset = 50

    fig, axs = plt.subplots(1, 3, figsize=(17, 8))

    # Make xy plot
    axs[0] = uct.plot_xy(
        pred_mean, pred_std, y, x, n_subset=300, ylims=ylims, xlims=[0, 15], ax=axs[0]
    )

    # Make ordered intervals plot
    axs[1] = uct.plot_intervals_ordered(
        pred_mean, pred_std, y, n_subset=n_subset, ylims=ylims, ax=axs[1]
    )

    # Make calibration plot
    axs[2] = uct.plot_calibration(pred_mean, pred_std, y, ax=axs[2])

    # Adjust subplots spacing
    fig.subplots_adjust(wspace=0.25)

    # Save figure
    if savefig:
        uct.viz.save_figure(plot_save_str, "svg", white_background=True)


# List of predictive means and standard deviations
pred_mean_list = [f]

pred_std_list = [
    std * 0.5,  # overconfident
    std * 2.0,  # underconfident
    std,  # correct
]

# Loop through, make plots, and compute metrics
idx_counter = 0
for i, pred_mean in enumerate(pred_mean_list):
    for j, pred_std in enumerate(pred_std_list):
        mace = uct.mean_absolute_calibration_error(pred_mean, pred_std, y)
        rmsce = uct.root_mean_squared_calibration_error(pred_mean, pred_std, y)
        ma = uct.miscalibration_area(pred_mean, pred_std, y)

        idx_counter += 1
        make_plots(pred_mean, pred_std, f"row_{idx_counter}")

        print(f"MACE: {mace}, RMSCE: {rmsce}, MA: {ma}")

Saved figure row_1.svg
MACE: 0.2796705308114705, RMSCE: 0.31469336905488715, MA: 0.28249548566815197
Saved figure row_2.svg
MACE: 0.1315802318486882, RMSCE: 0.1511902140132227, MA: 0.132909325099685
Saved figure row_3.svg
MACE: 0.0915924344112264, RMSCE: 0.10092580172498483, MA: 0.09251761051639032


In [38]:
# Save figure (set to True to save)
savefig = True

# List of predictive means and standard deviations
pred_mean_list = [f]
pred_std_list = [
    std * 0.5,  # overconfident
    std,
    std * 2.0,  # underconfident
]

# ylims for xy plot
ylims_xy = (-2.51, 3.31)

# Loop through, make plots, and compute metrics
for i, pred_mean in enumerate(pred_mean_list):
    for j, pred_std in enumerate(pred_std_list):
        # Before recalibration
        exp_props, obs_props = uct.get_proportion_lists_vectorized(
            pred_mean, pred_std, y
        )
        mace = uct.mean_absolute_calibration_error(
            pred_mean, pred_std, y, recal_model=None
        )
        rmsce = uct.root_mean_squared_calibration_error(
            pred_mean, pred_std, y, recal_model=None
        )
        ma = uct.miscalibration_area(pred_mean, pred_std, y, recal_model=None)
        print("Before Recalibration:  ", end="")
        print("MACE: {:.5f}, RMSCE: {:.5f}, MA: {:.5f}".format(mace, rmsce, ma))

        fig, axes = plt.subplots(1, 2, figsize=(11, 5))
        uct.plot_calibration(
            pred_mean,
            pred_std,
            y,
            exp_props=exp_props,
            obs_props=obs_props,
            ax=axes.flatten()[0],
        )
        uct.plot_xy(
            pred_mean,
            pred_std,
            y,
            x,
            ax=axes.flatten()[1],
            ylims=ylims_xy,
        )

        uct.viz.save_figure(f"before_recal_{j}", "png")

        # After recalibration
        std_recalibrator = uct.recalibration.get_std_recalibrator(
            pred_mean, pred_std, y
        )
        pred_std_recal = std_recalibrator(pred_std)

        mace = uct.mean_absolute_calibration_error(pred_mean, pred_std_recal, y)
        rmsce = uct.root_mean_squared_calibration_error(pred_mean, pred_std_recal, y)
        ma = uct.miscalibration_area(pred_mean, pred_std_recal, y)
        print("After Recalibration:  ", end="")
        print("MACE: {:.5f}, RMSCE: {:.5f}, MA: {:.5f}".format(mace, rmsce, ma))

        fig, axes = plt.subplots(1, 2, figsize=(11, 5))
        uct.plot_calibration(
            pred_mean,
            pred_std_recal,
            y,
            ax=axes.flatten()[0],
        )
        uct.plot_xy(
            pred_mean,
            pred_std_recal,
            y,
            x,
            ax=axes.flatten()[1],
            ylims=ylims_xy,
        )

        uct.viz.save_figure(f"after_recal_{j}", "png")

Before Recalibration:  MACE: 0.27967, RMSCE: 0.31469, MA: 0.28250
Saved figure before_recal_0.png
After Recalibration:  MACE: 0.01706, RMSCE: 0.02028, MA: 0.01721
Saved figure after_recal_0.png
Before Recalibration:  MACE: 0.09159, RMSCE: 0.10093, MA: 0.09252
Saved figure before_recal_1.png
After Recalibration:  MACE: 0.01706, RMSCE: 0.02040, MA: 0.01722
Saved figure after_recal_1.png
Before Recalibration:  MACE: 0.13158, RMSCE: 0.15119, MA: 0.13291
Saved figure before_recal_2.png
After Recalibration:  MACE: 0.01704, RMSCE: 0.02021, MA: 0.01720
Saved figure after_recal_2.png
